F)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.ticker import MultipleLocator

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error

from NeuralNetwork import NeuralNetwork
from Scheduler import *
from functions import *

import _LinearRegression
import OLS

import seaborn as sns

In [2]:
from sklearn.datasets import fetch_openml

# Fetch the MNIST dataset
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

# Extract data (features) and target (labels)
X = mnist.data
y = mnist.target

In [ ]:
def accuracy(pred, target):
    return np.mean(pred == target)

def dummy(x):
    return None

def one_hot_rep(data):
    out = np.zeros((np.shape(data)[0], 10))
    for i, point in enumerate(data):
        out[i, int(point)] = 1
    return out

In [196]:
# Scaling
X = X/255

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2025)

scaler_x = StandardScaler()
scaler_x.fit(X_train)
X_train_s = scaler_x.transform(X_train)
X_test_s = scaler_x.transform(X_test)

y_train = y_train.astype("int")
y_test = y_test.astype("int")

# Create one hot vectors
y_train_oh = one_hot_rep(y_train.ravel())
y_test_oh = one_hot_rep(y_test.ravel())

In [208]:
np.random.seed(2025)

num_input_feats = X_train.shape[1] # Number of input features
num_output_feats = 10 # Number og output features

batch_size = 1000
epochs = 50

np.random.seed(2025)

scheduler = Adam(0.001)
ffnn = NeuralNetwork(num_input_feats, 
                    num_output_feats,
                    [50, 50],
                    [ReLU, ReLU, softmax], 
                    [ReLU_der, ReLU_der, id_func_der],
                    scheduler,
                    cost_func= dummy,
                    cost_func_der= dummy,
                    classifier_mode=True,
                    regularization=None, 
                    reg_param=0,
                    seed=2025
                    )

#Train
ffnn.train(X_train_s, y_train_oh, batch_size, epochs)

In [207]:
# Predict
train_pred = ffnn.predict(X_train_s)
test_pred = ffnn.predict(X_test_s)
predict_train_labels = np.argmax(train_pred, axis=1)
predict_test_labels = np.argmax(test_pred, axis=1)
print("MSE train", accuracy(predict_train_labels, y_train))
print("MSE test:", accuracy(predict_test_labels, y_test))

MSE train 0.8950892857142857
MSE test: 0.8664285714285714
